<a href="https://colab.research.google.com/github/Caory2/Teoria_de_Aprendizaje_de_Maquina/blob/main/DOOM_PRIMER__NIVEL2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

caro83_doom_training_phase2_path = kagglehub.dataset_download('caro83/doom-training-phase2')

print('Data source import complete.')


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!sudo apt-get update && sudo apt-get install -y cmake libboost-all-dev libsdl2-dev libfreetype6-dev libgl1-mesa-dev libglu1-mesa-dev libpng-dev libjpeg-dev libbz2-dev libfluidsynth-dev libgme-dev libopenal-dev zlib1g-dev timidity tar nasm
!pip install "numpy<2.0" vizdoom stable-baselines3[extra] shimmy gymnasium imageio

In [ ]:
import os
import shutil

# --- CONFIGURACIÓN ---
# Asegúrate que este nombre coincida con el dataset que acabas de crear
INPUT_DATASET_DIR = "/kaggle/input/doom-training-phase2"
MODEL_FILENAME = "doom_ultimate_killer.zip" # El nombre del archivo que subiste
OUTPUT_DIR = "/kaggle/working"

print("🕵️‍♂️ Operación Rescate: Moviendo el modelo de 200k pasos...")

# Buscamos el archivo. Puede estar comprimido o descomprimido por Kaggle.
source_zip = os.path.join(INPUT_DATASET_DIR, MODEL_FILENAME)
source_unzipped = os.path.join(INPUT_DATASET_DIR, MODEL_FILENAME.replace(".zip", ""))

if os.path.exists(source_zip):
    print("✅ Archivo ZIP encontrado. Copiando...")
    shutil.copy(source_zip, os.path.join(OUTPUT_DIR, MODEL_FILENAME))

elif os.path.exists(os.path.join(INPUT_DATASET_DIR, "policy.pth")):
    print("⚠️ Kaggle descomprimió el archivo. Re-empaquetando...")
    shutil.make_archive(os.path.join(OUTPUT_DIR, "doom_ultimate_killer"), 'zip', INPUT_DATASET_DIR)

else:
    # Búsqueda profunda por si acaso
    print("🔍 Buscando en subcarpetas...")
    found = False
    for root, dirs, files in os.walk(INPUT_DATASET_DIR):
        if "policy.pth" in files:
            print(f"   Encontrado en: {root}. Re-empaquetando...")
            shutil.make_archive(os.path.join(OUTPUT_DIR, "doom_ultimate_killer"), 'zip', root)
            found = True
            break
    if not found:
        print("❌ ERROR CRÍTICO: No encuentro el modelo anterior en Input.")

In [ ]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import vizdoom as vzd
import cv2
import math
import os
import requests
import torch
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.callbacks import CheckpointCallback

# 1. Configuración Básica
print(f"🔍 GPU: {torch.cuda.is_available()}")
device_target = "cuda" if torch.cuda.is_available() else "cpu"

if not os.path.exists("doom1.wad"):
    url = "https://archive.org/download/2020_03_22_DOOM/DOOM1.WAD"
    r = requests.get(url, allow_redirects=True)
    with open("doom1.wad", "wb") as f:
        f.write(r.content)

# Configuración VizDoom
e1m1_config = """
doom_scenario_path = doom1.wad
doom_map = map01
living_reward = -0.01
episode_timeout = 50000
screen_resolution = RES_320X240
screen_format = RGB24
render_hud = false
render_crosshair = false
render_weapon = true
render_decals = false
render_particles = false
window_visible = false
available_game_variables = { HEALTH AMMO2 KILLCOUNT POSITION_X POSITION_Y }
available_buttons = { MOVE_LEFT MOVE_RIGHT ATTACK MOVE_FORWARD TURN_LEFT TURN_RIGHT USE }
mode = PLAYER
"""
with open("custom_ultimate_v2.cfg", "w") as f:
    f.write(e1m1_config)

# 2. Clase del Entorno con CASTIGOS DUROS
class DoomUltimateEnvV2(gym.Env):
    def __init__(self, config_file):
        super().__init__()
        self.game = vzd.DoomGame()
        self.game.load_config(config_file)
        self.game.set_screen_format(vzd.ScreenFormat.RGB24)
        self.game.set_screen_resolution(vzd.ScreenResolution.RES_320X240)
        self.game.init()
        self.action_space = spaces.Discrete(7)
        self.observation_space = spaces.Box(low=0, high=255, shape=(84, 84, 1), dtype=np.uint8)
        self.start_x = 0; self.start_y = 0; self.max_dist = 0; self.last_kills = 0; self.last_x = 0; self.last_y = 0

    def process_observation(self, state):
        if state is None: return np.zeros((84, 84, 1), dtype=np.uint8)
        screen = np.array(state.screen_buffer, copy=True)
        if screen is None or screen.size == 0: return np.zeros((84, 84, 1), dtype=np.uint8)
        try: screen = cv2.resize(screen, (84, 84))
        except: return np.zeros((84, 84, 1), dtype=np.uint8)
        if len(screen.shape) == 3 and screen.shape[2] == 3:
            screen = cv2.cvtColor(screen, cv2.COLOR_RGB2GRAY)
        return np.expand_dims(screen, axis=-1).astype(np.uint8)

    def step(self, action):
        buttons = [0]*7; buttons[action] = 1
        self.game.make_action(buttons, 4)
        state = self.game.get_state(); done = self.game.is_episode_finished()
        if done: return np.zeros((84, 84, 1), dtype=np.uint8), 0.0, True, False, {}

        vars = state.game_variables
        curr_x, curr_y = vars[3], vars[4]
        dist = math.sqrt((curr_x - self.start_x)**2 + (curr_y - self.start_y)**2)

        # --- NUEVA LÓGICA DE RECOMPENSAS ---
        # Exploración
        r_explore = (dist - self.max_dist) * 0.05 if dist > self.max_dist else 0
        if dist > self.max_dist: self.max_dist = dist

        # Kills
        r_kill = 50.0 if vars[2] > self.last_kills else 0

        # PAREDES (El cambio clave): Castigo brutal (-2.0) si se atasca
        r_stuck = 0
        if action in [0, 1, 3]: # Intentó moverse
            moved_dist = math.sqrt((curr_x - self.last_x)**2 + (curr_y - self.last_y)**2)
            if moved_dist < 1.5:
                r_stuck = -2.0 # <--- ¡AQUÍ ESTÁ LA MAGIA!

        total = r_explore + r_kill + r_stuck + self.game.get_last_reward()

        self.last_x, self.last_y = curr_x, curr_y
        self.last_kills = vars[2]
        return self.process_observation(state), float(total), done, False, {}

    def reset(self, seed=None):
        super().reset(seed=seed); self.game.new_episode(); state = self.game.get_state()
        if state:
            self.start_x = state.game_variables[3]; self.start_y = state.game_variables[4]
            self.last_x = self.start_x; self.last_y = self.start_y
            self.max_dist = 0; self.last_kills = 0
            return self.process_observation(state), {}
        return np.zeros((84, 84, 1), dtype=np.uint8), {}
    def close(self): self.game.close()

print("✅ Entorno V2 Listo: Las paredes ahora queman (-2.0 pts).")

In [ ]:
# --- FASE 2: ENTRENAMIENTO CORRECTIVO ---

env = DummyVecEnv([lambda: DoomUltimateEnvV2("custom_ultimate_v2.cfg")])
MODELO_ANTERIOR = "doom_ultimate_killer" # El zip que trajimos del Input

if os.path.exists(MODELO_ANTERIOR + ".zip"):
    print(f"🧠 Cargando modelo de 200k pasos: {MODELO_ANTERIOR}...")
    model = PPO.load(MODELO_ANTERIOR, env=env, device=device_target)

    # ESTRATEGIA DE RE-ENTRENAMIENTO:
    # 1. Learning Rate: 0.0001 (Ni muy rápido, ni muy lento).
    #    Queremos que cambie sus hábitos, pero no que olvide cómo ver.
    model.learning_rate = 0.0001

    # 2. Entropía: 0.1 (ALTA)
    #    Vital para que deje de hacer siempre lo mismo (vibrar en la pared).
    #    Lo obligará a probar acciones arriesgadas (como salir al pasillo).
    model.ent_coef = 0.1

    print("🚀 Iniciando Re-entrenamiento (Corrigiendo miedo a explorar)...")

    # Entrenamos por otros 100k - 200k pasos.
    # El agente sufrirá al principio por los castigos de pared,
    # pero luego encontrará la salida.
    model.learn(total_timesteps=500000)

    print("🏆 ¡Fase 2 Completada!")
    model.save("doom_agente_valiente")

else:
    print("❌ ERROR: No se encontró el modelo en el directorio de trabajo.")
    print("Revisa el BLOQUE 1.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
import os

# --- CONFIGURACIÓN ---
# Asegúrate de usar el nombre de tu MEJOR modelo (el de la Fase 2 o el Ultimate)
NOMBRE_MODELO = "doom_agente_valiente" # O "doom_ultimate_killer" si no hiciste la fase 2
NUM_EPISODIOS = 10  # Probamos 10 partidas para tener datos sólidos

# Verificamos si existe el modelo antes de empezar
if not os.path.exists(NOMBRE_MODELO + ".zip"):
    print(f"⚠️ No encontré '{NOMBRE_MODELO}'. Usando 'doom_ultimate_killer' por defecto.")
    NOMBRE_MODELO = "doom_ultimate_killer"

# Instanciamos el entorno (Usamos la V2 si la definiste, o la normal)
try:
    env = DummyVecEnv([lambda: DoomUltimateEnvV2("custom_ultimate_v2.cfg")])
    print("✅ Usando Entorno V2 (Castigos Duros)")
except:
    env = DummyVecEnv([lambda: DoomUltimateEnv("custom_ultimate.cfg")])
    print("✅ Usando Entorno V1 (Normal)")

# --- FUNCIONES DE EVALUACIÓN ---

def evaluar_agente(modelo, es_random=False, episodios=5):
    puntajes = []
    tiempos = [] # Cuántos frames sobrevivió

    for i in range(episodios):
        obs = env.reset()
        done = False
        total_reward = 0
        steps = 0

        while not done:
            if es_random:
                action = [env.action_space.sample()]
            else:
                action, _ = modelo.predict(obs, deterministic=True)

            obs, reward, done, _ = env.step(action)
            total_reward += reward[0]
            steps += 1

        puntajes.append(total_reward)
        tiempos.append(steps)

    return puntajes, tiempos

print(f"\n🧪 Evaluando Agente ALEATORIO ({NUM_EPISODIOS} partidas)...")
scores_random, times_random = evaluar_agente(None, es_random=True, episodios=NUM_EPISODIOS)

print(f"🤖 Evaluando Agente ENTRENADO ({NUM_EPISODIOS} partidas)...")
model = PPO.load(NOMBRE_MODELO)
scores_ppo, times_ppo = evaluar_agente(model, es_random=False, episodios=NUM_EPISODIOS)

# --- GENERACIÓN DE LA GRÁFICA ---

# Datos estadísticos
means = [np.mean(scores_random), np.mean(scores_ppo)]
maxs = [np.max(scores_random), np.max(scores_ppo)]
std_devs = [np.std(scores_random), np.std(scores_ppo)]
labels = ['Aleatorio (Baseline)', 'PPO (Entrenado)']

# Configuración del gráfico
plt.figure(figsize=(10, 6))

# Barras de Promedio con Error (Desviación Estándar)
bars = plt.bar(labels, means, yerr=std_devs, capsize=10, color=['#95a5a6', '#2ecc71'], alpha=0.7, label='Promedio')

# Puntos para el MEJOR puntaje (Récord)
plt.scatter(labels, maxs, color='red', zorder=5, s=100, label='Mejor Puntaje (Récord)')

# Etiquetas de valor sobre las barras
for bar, mean_val, max_val in zip(bars, means, maxs):
    # Texto del promedio
    plt.text(bar.get_x() + bar.get_width()/2, mean_val, f'{mean_val:.1f}',
             ha='center', va='bottom', fontweight='bold', color='black')
    # Texto del récord
    plt.text(bar.get_x() + bar.get_width()/2, max_val + (max_val*0.05), f'Máx: {max_val:.1f}',
             ha='center', va='bottom', color='red', fontsize=9)

plt.title(f'Comparación de Rendimiento: Agente Aleatorio vs PPO\n(Basado en {NUM_EPISODIOS} episodios)', fontsize=14)
plt.ylabel('Recompensa Total Acumulada', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.3)
plt.legend()

# Guardar y Mostrar
plt.savefig('comparativa_final_profesional.png', dpi=300)
print("\n✅ Gráfica guardada como 'comparativa_final_profesional.png'.")
plt.show()

# --- REPORTE DE TEXTO ---
print("-" * 40)
print("RESUMEN DE RESULTADOS:")
print(f"🎲 Random Promedio: {np.mean(scores_random):.2f} | Máximo: {np.max(scores_random):.2f}")
print(f"🤖 PPO Promedio:    {np.mean(scores_ppo):.2f} | Máximo: {np.max(scores_ppo):.2f}")

diff = np.mean(scores_ppo) - np.mean(scores_random)
print(f"\n📈 MEJORA: El agente entrenado obtiene {diff:.2f} puntos más que el azar.")
if np.mean(times_ppo) > np.mean(times_random):
    print(f"⏱️ Además, sobrevive {np.mean(times_ppo) - np.mean(times_random):.0f} frames más en promedio.")

In [ ]:
import imageio
import numpy as np
import os
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv

# --- Función Principal del Torneo ---
def grabar_mejor_partida_hasta_muerte(modelo, nombre_archivo, num_intentos=5, es_aleatorio=False):
    print(f"\n🎬 INICIANDO TORNEO DE SUPERVIVENCIA: {nombre_archivo}")
    print(f"   Se jugarán {num_intentos} partidas completas (Hasta morir).")
    print(f"   Se guardará SOLO la mejor.")

    # Intentamos cargar el entorno V2 (Castigos duros) si existe, sino el normal
    try:
        env = DummyVecEnv([lambda: DoomUltimateEnvV2("custom_ultimate_v2.cfg")])
        print("   ✅ Usando Entorno V2 (Hostil)")
    except:
        # Fallback al entorno normal si no definiste el V2 en esta sesión
        env = DummyVecEnv([lambda: DoomUltimateEnv("custom_ultimate.cfg")])
        print("   ✅ Usando Entorno V1 (Normal)")

    mejor_recompensa = -float('inf')
    mejores_frames = []
    mejor_duracion = 0

    for i in range(num_intentos):
        obs = env.reset()
        done = False
        recompensa_total = 0
        frames_actuales = []

        # BUCLE INFINITO (Hasta que muera 'done')
        while not done:
            # 1. Captura de pantalla
            screen = env.envs[0].game.get_state().screen_buffer
            if screen.shape[0] == 3: # Corregir canales (C, H, W -> H, W, C)
                screen = np.moveaxis(screen, 0, -1)
            frames_actuales.append(screen)

            # 2. Decisión del Agente
            if es_aleatorio:
                action = [env.action_space.sample()]
            else:
                # Deterministic=False para que use su 'instinto' y no se quede quieto
                action, _ = modelo.predict(obs, deterministic=False)

            # 3. Ejecutar acción
            obs, reward, done, _ = env.step(action)
            recompensa_total += reward[0]

        # Fin de la partida
        print(f"   💀 Partida {i+1}: Terminó en {len(frames_actuales)} frames | Puntos: {recompensa_total:.2f}")

        # 4. ¿Es este el nuevo campeón?
        # Usamos la recompensa total como criterio. Si prefieres duración, cambia a len(frames_actuales)
        if recompensa_total > mejor_recompensa:
            mejor_recompensa = recompensa_total
            mejores_frames = frames_actuales
            mejor_duracion = len(frames_actuales)
            print(f"      🌟 ¡NUEVO RÉCORD! (Guardando esta partida)")

    env.close()

    # 5. Guardar Video Final
    if len(mejores_frames) > 0:
        print(f"💾 Guardando LA MEJOR PARTIDA ({mejor_duracion} frames) en {nombre_archivo}...")
        imageio.mimsave(nombre_archivo, mejores_frames, fps=35)
        print("✅ Video generado exitosamente.")
    else:
        print("❌ Error: No se grabaron frames.")

# ==========================================
# EJECUCIÓN DEL TORNEO
# ==========================================

# 1. GRABAR AL AGENTE ALEATORIO
# -----------------------------
grabar_mejor_partida_hasta_muerte(None, "video_vs_random_death.mp4", num_intentos=3, es_aleatorio=True)

# 2. GRABAR A TU AGENTE ENTRENADO
# -----------------------------
# Buscamos cuál es tu mejor modelo disponible automáticamente
nombres_posibles = ["doom_agente_valiente", "doom_ultimate_killer", "doom_e1m1_campeon_final"]
model_path = None

for nombre in nombres_posibles:
    if os.path.exists(nombre + ".zip"):
        model_path = nombre
        break

if model_path:
    print(f"\n🤖 Cargando tu mejor modelo encontrado: {model_path}")
    model = PPO.load(model_path)

    # Jugamos 5 partidas y guardamos la mejor donde luchó hasta el final
    grabar_mejor_partida_hasta_muerte(model, "video_vs_entrenado_BEST_DEATH.mp4", num_intentos=5, es_aleatorio=False)
else:
    print("❌ No encontré ningún modelo entrenado (.zip). Verifica el nombre o la carpeta Output.")

In [ ]:
import imageio
import numpy as np
import os
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv

def grabar_muerte_epica(nombre_archivo, max_frames=10000): # Límite de seguridad de frames
    print(f"\n🎥 PREPARANDO GRABACIÓN SIN LÍMITE DE TIEMPO...")

    # 1. Buscamos el mejor modelo disponible
    nombres_posibles = ["doom_agente_valiente", "doom_ultimate_killer", "doom_e1m1_campeon_final"]
    model_path = None
    for nombre in nombres_posibles:
        if os.path.exists(nombre + ".zip"):
            model_path = nombre
            break

    if not model_path:
        print("❌ Error: No encuentro ningún modelo entrenado.")
        return

    print(f"🤖 Agente seleccionado: {model_path}")
    model = PPO.load(model_path)

    # 2. Cargamos el entorno (Leerá el nuevo config de 25 mins)
    # Intentamos cargar la clase V2, si falla usamos la normal
    try:
        env = DummyVecEnv([lambda: DoomUltimateEnvV2("custom_ultimate_v2.cfg")])
    except:
        env = DummyVecEnv([lambda: DoomUltimateEnv("custom_ultimate.cfg")])

    # 3. Iniciamos la grabación
    obs = env.reset()
    done = False
    frames = []
    total_reward = 0

    print("🔴 GRABANDO... (Esto durará hasta que la salud llegue a 0)")

    while not done and len(frames) < max_frames:
        # Captura
        screen = env.envs[0].game.get_state().screen_buffer
        if screen.shape[0] == 3: screen = np.moveaxis(screen, 0, -1)
        frames.append(screen)

        # Acción (Sin determinismo para que se mueva natural)
        action, _ = model.predict(obs, deterministic=False)
        obs, reward, done, _ = env.step(action)
        total_reward += reward[0]

        # Feedback cada 500 frames para saber que sigue vivo
        if len(frames) % 500 == 0:
            print(f"   ...Sigue vivo (Frame {len(frames)})...")

    env.close()

    print(f"💀 EL AGENTE MURIÓ (o llegó al límite de seguridad).")
    print(f"⏱️ Duración: {len(frames)} frames.")
    print(f"🏆 Puntos totales: {total_reward:.2f}")

    # Guardar MP4
    print(f"💾 Guardando video en {nombre_archivo}...")
    imageio.mimsave(nombre_archivo, frames, fps=35)
    print("✅ ¡Video Listo! Descárgalo en Output.")

# ¡ACCIÓN!
grabar_muerte_epica("video_muerte_final.mp4")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.monitor import Monitor # <--- IMPORTANTE
import os

# 1. Crear entorno con MONITOR
# El Monitor guarda los datos en 'log_dir'
log_dir = "/kaggle/working/logs/"
os.makedirs(log_dir, exist_ok=True)

def make_env():
    env = DoomUltimateEnvV2("custom_ultimate_v2.cfg") # O tu clase de entorno
    env = Monitor(env, log_dir) # <--- Aquí conectamos el grabador de datos
    return env

env = DummyVecEnv([make_env])

# 2. Entrenar (aunque sea un poco para probar)
print("Recopilando datos para la gráfica...")
model = PPO("CnnPolicy", env, verbose=0)
model.learn(total_timesteps=10000) # Entrena 10k pasos

# 3. Leer y Graficar el CSV generado
from stable_baselines3.common.results_plotter import load_results, ts2xy

def plot_monitor_log(folder):
    # Cargar datos
    x, y = ts2xy(load_results(folder), 'timesteps')

    plt.figure(figsize=(10, 5))
    plt.plot(x, y, color='blue', alpha=0.6, linewidth=1)

    # Media móvil para suavizar la línea (Rolling Average)
    if len(y) > 0:
        y_mean = pd.Series(y).rolling(window=10).mean()
        plt.plot(x, y_mean, color='red', linewidth=2, label='Media Móvil (10 ep)')

    plt.xlabel('Pasos de Tiempo')
    plt.ylabel('Recompensa del Episodio')
    plt.title('Progreso del Entrenamiento')
    plt.legend()
    plt.grid(True)
    plt.savefig("curva_aprendizaje_monitor.png")
    plt.show()

plot_monitor_log(log_dir)